# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamzaviour/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane:** Content Opportunity Scoring (refresh prioritisation). Data: the 30k-row starter slice (`data/raw/content_refresh_anonymized.csv`, one row per page, trailing 90 days). All wording is observed / directional / decision-support.

In [1]:
import json, os, numpy as np, pandas as pd
os.makedirs("work/outputs", exist_ok=True)
if os.path.exists("../../data/raw/content_refresh_anonymized.csv"): os.chdir("../..")   # run from work/notebooks
os.makedirs("work/outputs", exist_ok=True)
PATH = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(PATH):                       # Colab without the repo cloned
    import urllib.request
    PATH = "content_refresh_anonymized.csv"
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Hamzaviour/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv", PATH)
df = pd.read_csv(PATH)
print(df.shape)

(30000, 44)


## 0. Two signal checks before the rule

A signal check = one bucket table with **n printed** and a one-word verdict. `trend_direction` is used here only as a *yardstick* (is the page losing impressions?) — it is never an input to the rule.

**Signal A (flag-linked: the CTR-fix logic).** Does CTR fall as average position gets worse? If it does not, "CTR below what the position predicts" means nothing.

In [2]:
pos = df[df.avg_position > 0].copy()           # avg_position == 0 means "no data"
pos["pos_bucket"] = pd.cut(pos.avg_position, [0, 3, 10, 20, 50, 1000],
                           labels=["1-3", "3-10", "10-20", "20-50", "50+"])
a = (pos.groupby("pos_bucket", observed=True)
        .agg(n=("ctr", "size"), median_impr=("impressions_90d", "median"),
             median_ctr_pct=("ctr", "median"),
             pooled_ctr_pct=("clicks_90d", lambda s: np.nan))
        .drop(columns="pooled_ctr_pct"))
a["pooled_ctr_pct"] = (pos.groupby("pos_bucket", observed=True).clicks_90d.sum()
                       / pos.groupby("pos_bucket", observed=True).impressions_90d.sum() * 100)
print(a.round(2))
mono = a.pooled_ctr_pct.is_monotonic_decreasing
verdict_A = "CONFIRMED" if mono else "MIXED"
print("\nSignal A verdict:", verdict_A, "(pooled CTR falls with worse position:", mono, ")")

                n  median_impr  median_ctr_pct  pooled_ctr_pct
pos_bucket                                                    
1-3          1141         74.0            0.00            0.49
3-10        11842       1184.0            0.16            0.35
10-20        7273        870.0            0.10            0.35
20-50        7225        807.0            0.03            0.15
50+          1314        219.5            0.00            0.04

Signal A verdict: CONFIRMED (pooled CTR falls with worse position: True )


Reading: pooled CTR (total clicks ÷ total impressions) falls as position gets worse, so a page's CTR compared with the norm for its position band is a usable "CTR-fix" signal. Caveat: the 1–3 bucket has few, low-volume pages, so read its median with care.

**Signal B (flag-linked: staleness behind the refresh flags).** Do staler pages lose impressions more often? Yardstick: share of pages with `trend_direction == "down"`.

In [3]:
df["stale_bucket"] = pd.cut(df.days_since_last_update, [-1, 30, 90, 180, 365, 10000],
                            labels=["0-30", "31-90", "91-180", "181-365", "365+"])
b = (df.groupby("stale_bucket", observed=True)
       .agg(n=("trend_direction", "size"),
            pct_down=("trend_direction", lambda s: (s == "down").mean() * 100),
            median_impr=("impressions_90d", "median")))
print(b.round(1))
base_down = (df.trend_direction == "down").mean() * 100
print(f"\nbase rate of down: {base_down:.1f}%")
spread = b.pct_down.max() - b.pct_down.min()
inc = b.pct_down.is_monotonic_increasing; dec = b.pct_down.is_monotonic_decreasing
verdict_B = "CONFIRMED" if (inc and spread >= 5) else "OPPOSITE" if (dec and spread >= 5) else "MIXED" if spread >= 3 else "FALSE"
print("Signal B verdict:", verdict_B, f"(spread {spread:.1f} pp, monotonic up={inc}, down={dec})")

                  n  pct_down  median_impr
stale_bucket                              
0-30          20480      51.1        470.0
31-90           175      58.9        510.0
91-180         9171      61.1       1692.0
181-365         169      46.7         16.0
365+              5      60.0          2.0

base rate of down: 54.2%
Signal B verdict: MIXED (spread 14.4 pp, monotonic up=False, down=False)


Reading: see the verdict line above. If staleness does **not** line up with decline, that is a useful negative — it means the rule below should *not* be built on "old = declining". I therefore build the rule on Signal A (CTR vs position) and do not use staleness as a score driver.

## 1. My rule and its reason code

**Plain words:** a page is worth a title/meta rewrite when it is already seen a lot, ranks on the first two pages (position ≤ 20), and gets fewer clicks than the typical page at that position. The score is the **estimated clicks left on the table**:

`score = impressions_90d × max(0, expected_ctr(position band) − ctr) / 100`

where `expected_ctr` is the pooled CTR of the position band from Signal A. Eligibility: `0 < avg_position ≤ 20` and `impressions_90d ≥ 1000`.

- **One reason code:** `ctr_below_position_norm`
- **Action label:** `rewrite_title_meta`
- **Inputs used:** `avg_position`, `ctr`, `impressions_90d`, `clicks_90d` only. No `trend_*`, no `is_declining`, no future window, no IDs as features.

## 2. Ranked queue (writes the CSV)

In [4]:
BANDS = [0, 3, 10, 20]; LAB = ["1-3", "3-10", "10-20"]
elig = df[(df.avg_position > 0) & (df.avg_position <= 20) & (df.impressions_90d >= 1000)].copy()
elig["pos_band"] = pd.cut(elig.avg_position, BANDS, labels=LAB)
norm = a.pooled_ctr_pct.loc[LAB]                 # expected CTR per band (from Signal A table)
elig["expected_ctr_pct"] = elig.pos_band.map(norm).astype(float)
elig["ctr_gap_pct"] = (elig.expected_ctr_pct - elig.ctr).clip(lower=0)
elig["score"] = elig.impressions_90d * elig.ctr_gap_pct / 100          # est. missed clicks / 90d
elig["reason_code"] = "ctr_below_position_norm"
elig["action"] = "rewrite_title_meta"
queue = (elig[elig.score > 0].sort_values("score", ascending=False)
         [["content_id", "client_id", "score", "reason_code", "action", "avg_position",
           "impressions_90d", "clicks_90d", "ctr", "expected_ctr_pct", "days_with_impressions",
           "content_type", "days_since_last_update", "sessions_90d", "search_volume"]].reset_index(drop=True))
queue.insert(0, "rank", queue.index + 1)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
json.dump({"n_pages": int(len(df)), "n_eligible": int(len(elig)), "n_queue": int(len(queue)),
           "verdict_signal_A_ctr_vs_position": verdict_A, "verdict_signal_B_staleness_vs_decline": verdict_B,
           "expected_ctr_pct_by_band": {k: round(float(v), 3) for k, v in norm.items()},
           "rule": "impressions_90d * max(0, expected_ctr(position band) - ctr)/100"},
          open("work/outputs/baseline_action_score_metrics.json", "w"), indent=2)
print(f"eligible {len(elig):,} of {len(df):,}; queue {len(queue):,}; wrote work/outputs/baseline_action_score.csv")
queue.head(10)

eligible 10,011 of 30,000; queue 6,814; wrote work/outputs/baseline_action_score.csv


,rank,content_id,client_id,score,reason_code,action,avg_position,impressions_90d,clicks_90d,ctr,expected_ctr_pct,days_with_impressions,content_type,days_since_last_update,sessions_90d,search_volume
0,1,content_8c19996aa890,client_4e07408562,1731.103271,ctr_below_position_norm,rewrite_title_meta,2.5,509252,785,0.15,0.489931,88,keyword article,20,571,70.0
1,2,content_8451fc6f034d,client_d029fa3a95,1251.673474,ctr_below_position_norm,rewrite_title_meta,2.3,272144,75,0.03,0.489931,88,keyword article,20,99,10.0
2,3,content_5fe46e04994d,client_4e07408562,1082.352518,ctr_below_position_norm,rewrite_title_meta,4.2,517715,741,0.14,0.349063,88,keyword article,104,520,1900.0
3,4,content_36ff89c8214e,client_19581e27de,882.527085,ctr_below_position_norm,rewrite_title_meta,7.3,295097,154,0.05,0.349063,88,keyword article,104,238,0.0
4,5,content_c8e9d6ab9013,client_19581e27de,728.418496,ctr_below_position_norm,rewrite_title_meta,9.7,208678,0,0.00,0.349063,88,keyword article,104,6,20.0
5,6,content_c84a0ab98e90,client_f369cb89fc,712.376016,ctr_below_position_norm,rewrite_title_meta,7.8,223271,70,0.03,0.349063,88,keyword article,20,87,0.0
6,7,content_e12868d1f396,client_4e07408562,628.686468,ctr_below_position_norm,rewrite_title_meta,2.9,149712,104,0.07,0.489931,88,keyword article,7,101,12100.0
7,8,content_4a6607efcb46,client_6208ef0f77,614.637494,ctr_below_position_norm,rewrite_title_meta,2.2,128068,17,0.01,0.489931,88,keyword article,104,87,0.0
8,9,content_cb112fce36be,client_19581e27de,585.926345,ctr_below_position_norm,rewrite_title_meta,5.6,309910,492,0.16,0.349063,88,keyword article,104,480,70.0
9,10,content_73c54f78c06a,client_f369cb89fc,532.903496,ctr_below_position_norm,rewrite_title_meta,4.7,213963,211,0.10,0.349063,88,keyword article,20,116,10.0


## 3. Top-10 review — action, why it is there, what would make it wrong

In [5]:
top = queue.head(10)
client_n = top.client_id.value_counts()
def wrong(r):
    w = []
    if r.days_with_impressions < 60: w.append(f"impressions came on only {int(r.days_with_impressions)} of 90 days, so the average position may hide a spike")
    if r.avg_position > 10: w.append(f"position {r.avg_position:.1f} is page 2, so ranking work may matter more than CTR")
    if r.avg_position <= 3: w.append("the 1-3 band holds few low-volume pages, so its norm is shaky")
    if r.sessions_90d > 3 * max(r.clicks_90d, 1): w.append(f"GA4 shows {int(r.sessions_90d)} sessions vs {int(r.clicks_90d)} GSC clicks, so clicks may undercount real visits")
    if pd.notna(r.search_volume) and r.search_volume < 100: w.append(f"target keyword volume is only {int(r.search_volume)}, so the {int(r.impressions_90d):,} impressions likely come from off-target queries")
    if r.days_since_last_update <= 30: w.append(f"updated {int(r.days_since_last_update)} days ago, so CTR may still be settling")
    if client_n[r.client_id] >= 3: w.append(f"{client_n[r.client_id]} of the top 10 are the same client, so a client-wide cause (site title template) may explain it")
    w.append("a SERP feature (snippet, ads, video) could explain the low CTR rather than the title")
    return "; ".join(w[:4])
for _, r in top.iterrows():
    print(f"#{r['rank']:>2} [{r.action}] {r.content_id}  pos {r.avg_position:.1f}, {int(r.impressions_90d):,} impr, "
          f"CTR {r.ctr:.2f}% vs norm {r.expected_ctr_pct:.2f}% -> ~{r.score:,.0f} clicks/90d")
    print(f"     why: CTR is below what position {r.avg_position:.1f} usually gets ({r.reason_code}).")
    print(f"     wrong if: {wrong(r)}.")

# 1 [rewrite_title_meta] content_8c19996aa890  pos 2.5, 509,252 impr, CTR 0.15% vs norm 0.49% -> ~1,731 clicks/90d
     why: CTR is below what position 2.5 usually gets (ctr_below_position_norm).
     wrong if: the 1-3 band holds few low-volume pages, so its norm is shaky; target keyword volume is only 70, so the 509,252 impressions likely come from off-target queries; updated 20 days ago, so CTR may still be settling; 3 of the top 10 are the same client, so a client-wide cause (site title template) may explain it.
# 2 [rewrite_title_meta] content_8451fc6f034d  pos 2.3, 272,144 impr, CTR 0.03% vs norm 0.49% -> ~1,252 clicks/90d
     why: CTR is below what position 2.3 usually gets (ctr_below_position_norm).
     wrong if: the 1-3 band holds few low-volume pages, so its norm is shaky; target keyword volume is only 10, so the 272,144 impressions likely come from off-target queries; updated 20 days ago, so CTR may still be settling; a SERP feature (snippet, ads, video) could explain the l

## 4. Weak picks + leakage check

In [6]:
weak = top[(top.sessions_90d > 3 * top.clicks_90d.clip(lower=1)) | (top.search_volume < 100) | (top.days_with_impressions < 60) | (top.avg_position > 10)]
print("Top-10 rows with at least one weak-pick flag:", len(weak), "of 10")
print(weak[["rank", "content_id", "avg_position", "impressions_90d", "clicks_90d", "sessions_90d", "search_volume"]].to_string(index=False))

used = ["avg_position", "ctr", "impressions_90d", "clicks_90d", "days_with_impressions"]
banned = [c for c in df.columns if c.startswith("trend_") or "last_30d" in c or "prev_30d" in c or "declin" in c]
assert not set(used) & set(banned), "a label-derived / window column leaked into the rule"
print("\nLeak check passed. Rule inputs:", used)
print("Never used:", banned)

Top-10 rows with at least one weak-pick flag: 8 of 10
 rank           content_id  avg_position  impressions_90d  clicks_90d  sessions_90d  search_volume
    1 content_8c19996aa890           2.5           509252         785           571           70.0
    2 content_8451fc6f034d           2.3           272144          75            99           10.0
    4 content_36ff89c8214e           7.3           295097         154           238            0.0
    5 content_c8e9d6ab9013           9.7           208678           0             6           20.0
    6 content_c84a0ab98e90           7.8           223271          70            87            0.0
    8 content_4a6607efcb46           2.2           128068          17            87            0.0
    9 content_cb112fce36be           5.6           309910         492           480           70.0
   10 content_73c54f78c06a           4.7           213963         211           116           10.0

Leak check passed. Rule inputs: ['avg_position', 'ctr'

Weak picks are listed above: the table shows which of the ten have GA4 sessions far above GSC clicks, a tiny target-keyword volume, a spiky impression history or a page-2 position. These are why the top-10 is decision-support, not an order to act on blindly.

Limits: (1) `expected_ctr` is learned from the same slice, so the norm is in-sample; (2) the starter slice is one 90-day snapshot, so there is no before/after test of whether a rewrite helps; (3) the baseline cannot see SERP features. The Week-5 model must beat this ranking.

## Self-check

- [x] Two signal checks with bucket tables and n printed; Signal A (CTR vs position) and Signal B (staleness) are both flag-linked; one-word verdict each
- [x] One rule: score, one reason code (`ctr_below_position_norm`), one action label (`rewrite_title_meta`)
- [x] Ranked queue written to `work/outputs/baseline_action_score.csv` from this notebook (stays out of git); metrics JSON written alongside
- [x] Ten reviewed rows with "wrong if" lines
- [x] No `trend_*`, window-derived or label columns used as inputs (asserted in section 4)
- [x] No client names, URLs or private queries